<a href="https://colab.research.google.com/github/gph05010/Medication-Info-Text-Classification/blob/main/01_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%88%EB%9F%AC%EC%98%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import math
import time

# 1. API 기본 정보
url = ""
service_key = ""

# 2. 기본 요청 설정
num_of_rows = 100
page_no = 1

params = {
    "serviceKey": service_key,
    "pageNo": page_no,
    "numOfRows": num_of_rows,
    "type": "json"   # 문서에 따라 "_type": "json"일 수도 있음
}

# 3. 1페이지 요청 테스트
response = requests.get(url, params=params)

print("요청 URL:")
print(response.url)

print("\n상태 코드:")
print(response.status_code)

data = response.json()

print("\n응답 header:")
print(data["header"])

print("\nbody 정보:")
print("pageNo:", data["body"]["pageNo"])
print("numOfRows:", data["body"]["numOfRows"])
print("totalCount:", data["body"]["totalCount"])

In [ ]:
import requests
import pandas as pd
import math
import time

url = ""
service_key = ""

num_of_rows = 100
total_count = 4763
total_pages = math.ceil(total_count / num_of_rows)

all_items = []
failed_pages = []

for page in range(1, total_pages + 1):
    success = False

    for attempt in range(1, 4):  # 최대 3번 재시도
        params = {
            "serviceKey": service_key,
            "pageNo": page,
            "numOfRows": num_of_rows,
            "type": "json"
        }

        response = requests.get(url, params=params, timeout=20)

        if response.status_code == 200:
            try:
                data = response.json()

                if data["header"]["resultCode"] == "00":
                    items = data["body"].get("items", [])

                    if isinstance(items, dict):
                        items = [items]

                    all_items.extend(items)
                    print(f"{page}/{total_pages} 페이지 수집 완료 - 누적 {len(all_items)}건")
                    success = True
                    break
                else:
                    print(f"{page}페이지 API 오류:", data["header"])

            except Exception as e:
                print(f"{page}페이지 JSON 변환 실패 / {attempt}번째 시도")
                print(response.text[:300])

        else:
            print(f"{page}페이지 요청 실패: {response.status_code} / {attempt}번째 시도")
            print(response.text[:300])

        time.sleep(1)

    if not success:
        failed_pages.append(page)
        print(f"{page}페이지 최종 실패")

    time.sleep(0.2)

df = pd.DataFrame(all_items)

print("수집 완료")
print("데이터 크기:", df.shape)
print("실패 페이지:", failed_pages)

In [ ]:
print(failed_pages)

In [ ]:
print("수집 행 수:", len(df))
print("API totalCount:", total_count)
print("itemSeq 고유 개수:", df["itemSeq"].nunique())

In [ ]:
df.to_csv("medicine_raw.csv", index=False, encoding="utf-8-sig")

In [ ]:
import os

save_dir = "/content/drive/MyDrive/해커톤"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "medicine_raw.csv")

df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("저장 완료:", save_path)

In [ ]:
print("행 수:", df.shape[0])
print("열 수:", df.shape[1])

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
print("전체 행 수:", len(df))
print("itemSeq 고유 개수:", df["itemSeq"].nunique())
print("itemSeq 중복 개수:", len(df) - df["itemSeq"].nunique())

| 컬럼명                   | 의미       | 모델 활용 여부 |
| --------------------- | -------- | -------- |
| `itemName`            | 의약품명     | 참고용      |
| `entpName`            | 업체명      | 참고용      |
| `itemSeq`             | 품목 기준 코드 | 중복 확인용   |
| `efcyQesitm`          | 효능       | 라벨: 효능   |
| `useMethodQesitm`     | 복용법      | 라벨: 복용법  |
| `atpnQesitm`          | 주의사항     | 라벨: 주의사항 |
| `intrcQesitm`         | 상호작용     | 라벨: 상호작용 |
| `seQesitm`            | 부작용      | 라벨: 부작용  |
| `depositMethodQesitm` | 보관법      | 라벨: 보관법  |
| `openDe`              | 공개일자     | 참고용      |
| `updateDe`            | 수정일자     | 참고용      |
| `itemImage`           | 이미지 URL  | 미사용      |
| `bizrno`              | 사업자등록번호  | 미사용      |


In [ ]:
import pandas as pd

# CSV 불러오기
df = pd.read_csv("/content/drive/MyDrive/해커톤/medicine_raw.csv")

# Excel로 저장
df.to_excel("/content/drive/MyDrive/해커톤/medicine_raw.xlsx", index=False)